# GraphCPP Progress Report #3 — GraphSAGE-4 vs GraphSAGE-6 vs TransformerConv-2 vs TransformerConv-4

Same environment as the Progress Report #2 notebook (Kaggle T4, Python 3.10, torch 1.13.1+cu117, PyG 2.3.0, pytorch-lightning 2.0.1.post0, rdkit 2022.09.5, deepchem 2.7.1).

| Experiment | layer_type | mp_layers |
|---|---|---:|
| `graphcpp_sage4` (PR2 reference, re-run for paired analysis) | sageconv | 4 |
| `graphcpp_sage6` (A) | sageconv | 6 |
| `graphcpp_transformer2` (B) | transformerconv (4 heads, concat=False, edge_dim=11) | 2 |
| `graphcpp_transformer4` (C) | transformerconv (4 heads, concat=False, edge_dim=11) | 4 |

Everything else is identical: seed 42, same train/val/test CSVs, preprocessing, hidden_channels=128, mean pooling, topological torsion fingerprint branch, Adam lr=0.001 + cosine schedule, 100 epochs, best checkpoint by validation MCC, threshold 0.5, PR2 length bins.

The `pr3-graph-transformer` branch already contains the compatibility fixes that the PR2 notebook applied at runtime, so no patching is needed here.

**Before running:** push the `pr3-graph-transformer` branch to GitHub (or change `REPO_BRANCH`). Enable a GPU and Internet.

In [ ]:
# 1) Platform, GPU, and Internet check
import os
import sys
import shutil
import subprocess
import urllib.request
from pathlib import Path

if Path("/kaggle/working").exists():
    WORK_ROOT = Path("/kaggle/working")
    PLATFORM = "Kaggle"
else:
    WORK_ROOT = Path("/content")
    PLATFORM = "Google Colab"

print("Platform:", PLATFORM)
print("Notebook Python:", sys.version.split()[0])
print("WORK_ROOT:", WORK_ROOT)

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
        text=True,
    )
    print("\nGPU:")
    print(gpu_info)
except Exception as e:
    raise RuntimeError(
        "No NVIDIA GPU is visible. Enable a GPU accelerator, reconnect, and Run All again."
    ) from e

try:
    with urllib.request.urlopen("https://github.com", timeout=15) as r:
        print("Internet access: OK")
except Exception as e:
    raise RuntimeError(
        "Internet access is unavailable. Enable Internet/network access and Run All again."
    ) from e

In [ ]:
# 2) Clean workspace, install Micromamba, and define helpers
ENV_PREFIX = WORK_ROOT / "graphcpp_env"
MAMBA_ROOT = WORK_ROOT / "micromamba_root"
BIN_DIR = WORK_ROOT / "bin"
MM_BIN = BIN_DIR / "micromamba"
PROJECT = WORK_ROOT / "GraphCPP_pr3"

for p in (ENV_PREFIX, MAMBA_ROOT, PROJECT):
    if p.exists():
        shutil.rmtree(p)

if MM_BIN.exists():
    MM_BIN.unlink()

BIN_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    f"curl -fLsS https://micro.mamba.pm/api/micromamba/linux-64/latest "
    f"| tar -xj -C {WORK_ROOT} bin/micromamba",
    shell=True,
    check=True,
)

if not MM_BIN.exists():
    raise RuntimeError("Micromamba installation failed.")

MAMBA_ENV = os.environ.copy()
MAMBA_ENV["MAMBA_ROOT_PREFIX"] = str(MAMBA_ROOT)

def run_checked(cmd, *, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd))
    subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        env=env,
        check=True,
    )

def pip_install(*args):
    run_checked([ENV_PYTHON, "-m", "pip", "install", *args], env=RUNTIME_ENV)

def env_python(*args, cwd=None):
    run_checked([ENV_PYTHON, *args], cwd=cwd, env=RUNTIME_ENV)

run_checked([MM_BIN, "--version"], env=MAMBA_ENV)

In [ ]:
# 3) Create isolated Python 3.10 environment
run_checked(
    [
        MM_BIN, "create", "-y",
        "-p", ENV_PREFIX,
        "-c", "conda-forge",
        "python=3.10",
        "pip",
    ],
    env=MAMBA_ENV,
)

ENV_PYTHON = str(ENV_PREFIX / "bin" / "python")
if not Path(ENV_PYTHON).exists():
    raise RuntimeError(f"Environment Python missing: {ENV_PYTHON}")

RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV["PATH"] = f"{ENV_PREFIX / 'bin'}:{RUNTIME_ENV['PATH']}"
RUNTIME_ENV["PYTHONNOUSERSITE"] = "1"

pip_install(
    "--upgrade",
    "pip==24.0",
    "setuptools==69.5.1",
    "wheel==0.43.0",
)

env_python("--version")

In [ ]:
# 4) PyTorch 1.13.1 + CUDA 11.7
pip_install(
    "torch==1.13.1+cu117",
    "torchvision==0.14.1+cu117",
    "torchaudio==0.13.1",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cu117",
)

In [ ]:
# 5) Scientific / chemistry stack
# Use Micromamba for the old scientific stack to avoid pip resolver problems.
run_checked(
    [
        MM_BIN, "install", "-y",
        "-p", ENV_PREFIX,
        "-c", "conda-forge",
        "numpy=1.24.4",
        "pandas=1.5.3",
        "scipy=1.10.1",
        "scikit-learn=1.2.2",
        "joblib=1.2.0",
        "sympy=1.11.1",
        "tqdm=4.65.0",
        "rdkit=2022.09.5",
        "python-dateutil=2.8.2",
        "pytz=2023.3",
    ],
    env=MAMBA_ENV,
)

# GraphCPP uses DeepChem's graph featurizer.
pip_install("--no-deps", "deepchem==2.7.1")

In [ ]:
# 6) PyTorch Lightning + PyTorch Geometric
pip_install(
    "pytorch-lightning==2.0.1.post0",
    "torchmetrics==0.11.4",
    "lightning-utilities==0.8.0",
)

pip_install("torch-geometric==2.3.0")

pip_install(
    "torch-scatter==2.1.1",
    "torch-sparse==0.6.17",
    "-f",
    "https://data.pyg.org/whl/torch-1.13.1+cu117.html",
)

In [ ]:
# 7) Clone the PR3 branch
REPO_URL = "https://github.com/finmook/GraphCPP.git"
REPO_BRANCH = "pr3-graph-transformer"

run_checked(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, PROJECT])
DATASET = PROJECT / "dataset" / "raw" / "2024-07-14-15-40-08"
assert (DATASET / "test.csv").is_file()
env_python("-m", "compileall", "-q", str(PROJECT), cwd=PROJECT)
print("PROJECT:", PROJECT)
print("DATASET:", DATASET)

In [ ]:
# 8) Sanity check: one real CPP1708 batch through every architecture (also processes the test split)
env_python("-c", "import torch; print('CUDA available:', torch.cuda.is_available()); assert torch.cuda.is_available()", cwd=PROJECT)
env_python("sanity_check.py", "--dataset", str(DATASET), cwd=PROJECT)

## Train the four models (seed 42, best checkpoint by validation MCC)

In [ ]:
# 9) Train
EXPERIMENTS = ["graphcpp_sage4", "graphcpp_sage6", "graphcpp_transformer2", "graphcpp_transformer4"]

for name in EXPERIMENTS:
    print("\n" + "=" * 100 + "\nTRAINING: " + name + "\n" + "=" * 100)
    log_path = PROJECT / f"{name}_run.log"
    with log_path.open("w", encoding="utf-8") as log_file:
        proc = subprocess.Popen(
            [ENV_PYTHON, "main.py", "--seed", "42", "--config", name, "--dataset", str(DATASET)],
            cwd=str(PROJECT), env=RUNTIME_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in proc.stdout:
            if "it/s" in line or "s/it" in line:
                continue
            log_file.write(line)
            if line.startswith(("Seed", "Experiment", "layer_type", "Parameter count", "Test MCC")) or "test_" in line:
                print(line, end="")
        if proc.wait() != 0:
            raise RuntimeError(f"{name} failed. See {log_path}")

## Evaluation: overall, PR2 length bins, paired same-peptide analysis

In [ ]:
# 10) Evaluate best checkpoints on the unshuffled test.csv and build the comparison tables
env_python("evaluate_pr3.py", cwd=PROJECT)

import pandas as pd
from IPython.display import display
R = PROJECT / "results"
for f in ["pr3_overall_comparison.csv", "pr3_length_comparison.csv", "pr3_paired_vs_sage4.csv"]:
    print(f); display(pd.read_csv(R / f))

# Copy results (+ run logs and best checkpoints) out of the project and zip them
OUT = WORK_ROOT / "pr3_results"
if OUT.exists():
    shutil.rmtree(OUT)
shutil.copytree(R, OUT)
shutil.copytree(PROJECT / "logs", OUT / "logs")
for log in PROJECT.glob("*_run.log"):
    shutil.copy(log, OUT / log.name)
shutil.make_archive(str(WORK_ROOT / "pr3_results"), "zip", root_dir=OUT)
print("Saved:", OUT, "and", WORK_ROOT / "pr3_results.zip")


In [ ]:
# 11) Remove the large environment/repo copies so the saved notebook Output only contains the results.
# (Run this last. Re-running training afterwards requires Run All again.)
for p in (ENV_PREFIX, MAMBA_ROOT, BIN_DIR, PROJECT):
    if p.exists():
        shutil.rmtree(p)
for p in WORK_ROOT.iterdir():
    print(p)

## How to interpret

- GraphSAGE-4 here is a **re-run** in the same session (needed for the paired analysis). Compare its overall MCC with PR2 (0.6389); GPU nondeterminism can cause small differences.
- 33–39 has only 14 peptides and 39–44 only 2 (both CPP, MCC undefined). One peptide changes 33–39 accuracy by ~7 pp.
- Only claim a long-peptide improvement if the 33–39 accuracy/paired counts actually show one.